# 6-Model Ensemble Data Extraction and Aggregation at Station Locations
## Reference Period: 1995–2014 | SSP2-4.5

**Purpose:**  
Extract the 6-model equal-weight ensemble precipitation from the Jordan-wide unified NetCDF file at each of the 49 observed station locations and aggregate to the same temporal scales used in Notebooks 01 and 06:

1. **Daily** time series (mm/day) — 1995–2014  
2. **Monthly totals** (mm/month) — calendar-year aggregation  
3. **Long-term monthly means** (mm/month) — 12-month climatology over 1995–2014

**Input NC file:** `Jordan_6model_ensemble_ssp245_1995_2014.nc` (Notebook 08)

**Outputs saved to:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6ensemble data\`

```
6ensemble data/
  daily/
    daily_6ens_all_stations.csv
  monthly/
    monthly_6ens_all_stations.csv
  long_term_monthly_mean/
    ltmm_6ens_all_stations.csv
```


## 1. Import Libraries

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr)]:
    print(f"  {lib:<12}: {mod.__version__}")


Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0


## 2. Configuration

In [2]:
# ── Input paths ───────────────────────────────────────────────────────────────
STATION_MAPPING_FILE = Path(
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

# 6-model unified ensemble NC for 1995-2014 (from Notebook 08)
ENSEMBLE_NC = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\6 ensemble models\Jordan_6model_ensemble_ssp245_1995_2014.nc"
)

# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6ensemble data"
)
DAILY_OUT   = OUTPUT_ROOT / "daily"
MONTHLY_OUT = OUTPUT_ROOT / "monthly"
LTMM_OUT    = OUTPUT_ROOT / "long_term_monthly_mean"

for d in [DAILY_OUT, MONTHLY_OUT, LTMM_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# ── Period ────────────────────────────────────────────────────────────────────
PERIOD_START = 1995
PERIOD_END   = 2014

MONTH_NAMES = {
    1:"January", 2:"February", 3:"March",    4:"April",
    5:"May",     6:"June",     7:"July",      8:"August",
    9:"September",10:"October",11:"November",12:"December"
}

PR_VAR = "prAdjust"

print("Configuration loaded.")
print(f"  Ensemble NC  : {ENSEMBLE_NC}")
print(f"  Output root  : {OUTPUT_ROOT}")
print(f"  Period       : {PERIOD_START}–{PERIOD_END}")


Configuration loaded.
  Ensemble NC  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6 ensemble models\Jordan_6model_ensemble_ssp245_1995_2014.nc
  Output root  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6ensemble data
  Period       : 1995–2014


## 3. Load Station Locations

In [3]:
stations = pd.read_excel(STATION_MAPPING_FILE)
stations["Station_ID"]   = stations["Station_ID"].astype(str).str.strip()
stations["Station_Name"] = stations["Station_Name"].astype(str).str.strip()
stations["Basin"]        = stations["Basin"].astype(str).str.strip()

print(f"Stations loaded : {len(stations)}")
print(f"Unique basins   : {stations['Basin'].nunique()}")
print()
print(stations[["Station_ID","Station_Name","Basin",
                "Latitude","Longitude"]].to_string(index=False))


Stations loaded : 49
Unique basins   : 12

Station_ID                        Station_Name                 Basin  Latitude  Longitude
    AB0004                      KH.EL-WAHADNEH               N.R.S.W 32.328239  35.643545
    AD0002                              HARTHA      YARMOUK (JORDAN) 32.692571  35.844729
    AD0005                             UM QEIS      YARMOUK (JORDAN) 32.654533  35.679241
    AD0008                              KHARJA      YARMOUK (JORDAN) 32.658084  35.887122
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN) 32.417193  35.911881
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN) 32.509924  36.199947
    AD0032     BAQURA MET.STATION (METEO DEPT) JORDAN VALLY (JORDAN) 32.612437  35.596982
    AE0003                           KAFR YUBA               N.R.S.W 32.543115  35.798953
    AE0004                           KAFR ASAD               N.R.S.W 32.600307  35.710913
    AH0002                       WADI EL-YABIS JORDAN VAL

## 4. Open Ensemble NetCDF and Match Stations to Grid

Identical k-d tree nearest-grid-point matching as used in Notebooks 01 and 06, ensuring consistent station-to-grid correspondence across all three ensemble approaches.


In [4]:
DISTANCE_THRESHOLD_KM = 25.0

ds_ens = xr.open_dataset(ENSEMBLE_NC)
lats   = ds_ens["lat"].values
lons   = ds_ens["lon"].values
times  = pd.to_datetime(ds_ens["time"].values)
n_lat, n_lon = len(lats), len(lons)

print(f"6-model ensemble NC opened.")
print(f"  Grid      : {n_lat} lat × {n_lon} lon")
print(f"  Time      : {times[0].date()} → {times[-1].date()}  ({len(times):,} steps)")
print(f"  Variables : {list(ds_ens.data_vars)}")
print()

# Build k-d tree
lon_grid, lat_grid = np.meshgrid(lons, lats)
grid_coords = np.column_stack([lat_grid.ravel(), lon_grid.ravel()])
lat_idx_flat, lon_idx_flat = np.unravel_index(
    np.arange(len(grid_coords)), lat_grid.shape
)
tree = cKDTree(grid_coords)

# Match stations
qc_rows = []
for _, stn in stations.iterrows():
    dist_deg, idx = tree.query([stn["Latitude"], stn["Longitude"]])
    li = int(lat_idx_flat[idx])
    lj = int(lon_idx_flat[idx])
    dist_km = dist_deg * 111.32
    qc_rows.append({
        "Station_ID"  : stn["Station_ID"],
        "Station_Name": stn["Station_Name"],
        "Basin"       : stn["Basin"],
        "Stn_Lat"     : round(stn["Latitude"],  6),
        "Stn_Lon"     : round(stn["Longitude"], 6),
        "Grid_Lat"    : round(lats[li], 4),
        "Grid_Lon"    : round(lons[lj], 4),
        "Lat_Idx"     : li,
        "Lon_Idx"     : lj,
        "Distance_km" : round(dist_km, 2),
        "Flag"        : "OK" if dist_km <= DISTANCE_THRESHOLD_KM else "DISTANT",
    })

qc_df = pd.DataFrame(qc_rows)
n_flagged = (qc_df["Flag"] == "DISTANT").sum()
print(f"Station-grid matching complete.")
print(f"  Matched   : {len(qc_df)}")
print(f"  Flagged (>{DISTANCE_THRESHOLD_KM} km) : {n_flagged}")
print()
print(qc_df[["Station_ID","Station_Name","Basin",
             "Grid_Lat","Grid_Lon","Distance_km","Flag"]].to_string(index=False))


6-model ensemble NC opened.
  Grid      : 85 lat × 75 lon
  Time      : 1995-01-01 → 2014-12-31  (7,305 steps)
  Variables : ['prAdjust', 'basin_id']

Station-grid matching complete.
  Matched   : 49
  Flagged (>25.0 km) : 0

Station_ID                        Station_Name                 Basin  Grid_Lat  Grid_Lon  Distance_km Flag
    AB0004                      KH.EL-WAHADNEH               N.R.S.W     32.35     35.65         2.53   OK
    AD0002                              HARTHA      YARMOUK (JORDAN)     32.65     35.85         4.78   OK
    AD0005                             UM QEIS      YARMOUK (JORDAN)     32.65     35.65         3.29   OK
    AD0008                              KHARJA      YARMOUK (JORDAN)     32.65     35.85         4.23   OK
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN)     32.45     35.95         5.60   OK
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN)     32.55     36.15         7.13   OK
    AD0032     BAQURA MET

## 5. Extract Daily 6-Model Ensemble Precipitation at Station Locations

The full precipitation array is loaded into memory once (~177 MB for the 1995–2014 slice), then each station's time series is extracted by indexing the matched grid cell. Values are already the 6-model ensemble mean — no further averaging is required.

Output: `daily/daily_6ens_all_stations.csv`  
Columns: `Date`, `Station_ID`, `Station_Name`, `Basin`, `pr_mm_day`


In [5]:
daily_csv = DAILY_OUT / "daily_6ens_all_stations.csv"

if daily_csv.exists():
    print(f"[SKIP] {daily_csv}")
    daily_df = pd.read_csv(daily_csv, parse_dates=["Date"])
else:
    pr_array = ds_ens[PR_VAR].values    # (n_time, n_lat, n_lon), float32

    rows = []
    for _, stn in tqdm(qc_df.iterrows(), total=len(qc_df), desc="Extracting stations"):
        li = int(stn["Lat_Idx"])
        lj = int(stn["Lon_Idx"])
        ts = pr_array[:, li, lj].astype(float)

        for date, val in zip(times, ts):
            rows.append({
                "Date"        : date.date(),
                "Station_ID"  : stn["Station_ID"],
                "Station_Name": stn["Station_Name"],
                "Basin"       : stn["Basin"],
                "pr_mm_day"   : round(float(val), 4) if np.isfinite(val) else np.nan,
            })

    daily_df = pd.DataFrame(rows)
    daily_df["Date"] = pd.to_datetime(daily_df["Date"])
    daily_df.sort_values(["Station_ID","Date"], inplace=True)
    daily_df.reset_index(drop=True, inplace=True)
    daily_df.to_csv(daily_csv, index=False)
    print(f"Daily CSV saved: {daily_csv}")

print(f"  Rows : {len(daily_df):,}")
print()
print("Preview:")
print(daily_df.head(6).to_string(index=False))


Extracting stations:   0%|          | 0/49 [00:00<?, ?it/s]

Daily CSV saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6ensemble data\daily\daily_6ens_all_stations.csv
  Rows : 357,945

Preview:
      Date Station_ID   Station_Name   Basin  pr_mm_day
1995-01-01     AB0004 KH.EL-WAHADNEH N.R.S.W     5.8323
1995-01-02     AB0004 KH.EL-WAHADNEH N.R.S.W     3.9886
1995-01-03     AB0004 KH.EL-WAHADNEH N.R.S.W     1.2666
1995-01-04     AB0004 KH.EL-WAHADNEH N.R.S.W     5.9882
1995-01-05     AB0004 KH.EL-WAHADNEH N.R.S.W     0.0057
1995-01-06     AB0004 KH.EL-WAHADNEH N.R.S.W     0.0024


## 6. Aggregate Daily to Monthly Totals

$$\text{MS}(y, m) = \sum_{d=1}^{n} P(y, m, d)$$

Output: `monthly/monthly_6ens_all_stations.csv`


In [6]:
monthly_csv = MONTHLY_OUT / "monthly_6ens_all_stations.csv"

if monthly_csv.exists():
    print(f"[SKIP] {monthly_csv}")
    monthly_df = pd.read_csv(monthly_csv)
else:
    daily_df["Year"]  = daily_df["Date"].dt.year
    daily_df["Month"] = daily_df["Date"].dt.month

    monthly_df = (
        daily_df
        .groupby(["Station_ID","Station_Name","Basin","Year","Month"], sort=True)
        ["pr_mm_day"]
        .sum()
        .reset_index()
        .rename(columns={"pr_mm_day": "pr_mm_month"})
    )
    monthly_df["pr_mm_month"] = monthly_df["pr_mm_month"].round(4)
    monthly_df.to_csv(monthly_csv, index=False)
    print(f"Monthly CSV saved: {monthly_csv}")

print(f"  Rows : {len(monthly_df):,}")


Monthly CSV saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6ensemble data\monthly\monthly_6ens_all_stations.csv
  Rows : 11,760


## 7. Compute Long-Term Monthly Means

$$\text{LTMA}(m) = \frac{1}{Y}\sum_{y=1}^{Y}\text{MS}(y, m), \quad Y = 20$$

Output: `long_term_monthly_mean/ltmm_6ens_all_stations.csv`  
Columns: `Model`, `Station_ID`, `Station_Name`, `Basin`, `Month`, `Month_Name`, `LTMM_mm_month`, `N_Years`


In [7]:
ltmm_csv = LTMM_OUT / "ltmm_6ens_all_stations.csv"

if ltmm_csv.exists():
    print(f"[SKIP] {ltmm_csv}")
    ltmm_df = pd.read_csv(ltmm_csv)
else:
    ltmm_df = (
        monthly_df
        .groupby(["Station_ID","Station_Name","Basin","Month"], sort=True)
        ["pr_mm_month"]
        .agg(LTMM_mm_month="mean", N_Years="count")
        .reset_index()
    )
    ltmm_df["LTMM_mm_month"] = ltmm_df["LTMM_mm_month"].round(4)
    ltmm_df["Month_Name"]    = ltmm_df["Month"].map(MONTH_NAMES)
    ltmm_df["Model"]         = "6-Model Ensemble"

    ltmm_df = ltmm_df[["Model","Station_ID","Station_Name","Basin",
                         "Month","Month_Name","LTMM_mm_month","N_Years"]]
    ltmm_df.to_csv(ltmm_csv, index=False)
    print(f"LTMM CSV saved: {ltmm_csv}")

print(f"  Rows : {len(ltmm_df):,}  "
      f"({ltmm_df['Station_ID'].nunique()} stations × 12 months)")
print()
print("Preview — first station (12 months):")
sid0 = ltmm_df["Station_ID"].iloc[0]
print(ltmm_df[ltmm_df["Station_ID"] == sid0]
      [["Station_ID","Station_Name","Basin","Month_Name",
        "LTMM_mm_month","N_Years"]].to_string(index=False))


LTMM CSV saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6ensemble data\long_term_monthly_mean\ltmm_6ens_all_stations.csv
  Rows : 588  (49 stations × 12 months)

Preview — first station (12 months):
Station_ID   Station_Name   Basin Month_Name  LTMM_mm_month  N_Years
    AB0004 KH.EL-WAHADNEH N.R.S.W    January        87.3077       20
    AB0004 KH.EL-WAHADNEH N.R.S.W   February        80.3805       20
    AB0004 KH.EL-WAHADNEH N.R.S.W      March        64.8953       20
    AB0004 KH.EL-WAHADNEH N.R.S.W      April        21.0483       20
    AB0004 KH.EL-WAHADNEH N.R.S.W        May         4.3854       20
    AB0004 KH.EL-WAHADNEH N.R.S.W       June         0.1003       20
    AB0004 KH.EL-WAHADNEH N.R.S.W       July         0.0102       20
    AB0004 KH.EL-WAHADNEH N.R.S.W     August         0.0083       20
    AB0004 KH.EL-WAHADNEH N.R.S.W  September         0.2520       20
    AB0004 KH.EL-WAHADNEH N.R.S.W    October        11.6859       20
    AB0004 KH.EL-WAHADNEH N.R

## 8. Output Summary

In [8]:
ds_ens.close()

print("=" * 65)
print("OUTPUT DIRECTORY STRUCTURE")
print("=" * 65)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    dirs[:] = sorted(d for d in dirs if not d.startswith("."))
    level   = Path(root).relative_to(OUTPUT_ROOT).parts
    indent  = "  " * len(level)
    print(f"{indent}📁 {Path(root).name}/")
    sub = "  " * (len(level) + 1)
    for f in sorted(files):
        sz = (Path(root) / f).stat().st_size / 1024
        unit = "KB"
        if sz > 1024:
            sz /= 1024; unit = "MB"
        print(f"{sub}📄 {f}  ({sz:.1f} {unit})")

print()
print("Next step → run Notebook 10_6model_Validation.ipynb")


OUTPUT DIRECTORY STRUCTURE
📁 6ensemble data/
  📁 daily/
    📄 daily_6ens_all_stations.csv  (17.1 MB)
  📁 long_term_monthly_mean/
    📄 ltmm_6ens_all_stations.csv  (40.3 KB)
  📁 monthly/
    📄 monthly_6ens_all_stations.csv  (546.4 KB)

Next step → run Notebook 10_6model_Validation.ipynb
